# Predicting Bitcoin Prices from Chainlet Data, Explained with SHAP

**CAP 5619, AI for Finance. Alex Hubenthal.**

This notebook predicts the daily Bitcoin price for every day of December 2017, one day ahead, from
lagged prices and chainlet features, and explains the model with SHAP. The data are the three files
provided with the course: the occurrence chainlets, the amount chainlets and the daily price.

Constraints the notebook is built around:

* Chainlet features come from both the occurrence file and the amount file.
* RMSE is computed in USD over the 31 days of December 2017.
* Dates are parsed with datetime functions and never sliced as strings. `year`, `day` and `date` are
  never model inputs.
* The horizon is one day, and nothing from December 2017 reaches training, scaling or model selection.
* Regression or feedforward networks only, no ensembles.
* SHAP explains at least three December predictions, each followed by a written reading.

## 0. Setup

The first cell checks the Python version and installs any missing package, so the notebook can start on
a machine with nothing installed. The second cell defines the constants once, so the test window and the
horizon cannot drift between cells.

In [1]:
import importlib
import importlib.util
import subprocess
import sys

MIN_PYTHON = (3, 9)
REQUIRED = {  # import name -> pip name
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "shap": "shap",
}

if sys.version_info < MIN_PYTHON:
    raise RuntimeError(
        "Python %d.%d or newer is required, found %d.%d. Install a newer Python and rerun."
        % (MIN_PYTHON + tuple(sys.version_info[:2]))
    )

missing = [pip for mod, pip in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", *missing],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        # An "externally managed environment" error means the notebook is running in the system Python.
        raise RuntimeError(
            "pip could not install %s. If it reports an externally managed environment, run the notebook "
            "from a virtual environment (python -m venv .venv).\n%s" % (", ".join(missing), result.stderr[-1500:])
        )
    importlib.invalidate_caches()

versions = {name: importlib.import_module(name).__version__ for name in REQUIRED}
print("Python %d.%d.%d | " % tuple(sys.version_info[:3]) + ", ".join("%s %s" % item for item in versions.items()))

Python 3.13.18 | numpy 2.5.5, pandas 3.1.1, sklearn 1.9.1, matplotlib 3.11.2, shap 0.54.1


In [2]:
import os
import urllib.request

import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 12)

SEED = 5619
np.random.seed(SEED)

# The three course files are read from disk and never embedded in the notebook.
DATA_DIR = "data"
FILES = {
    "occ": "OccChainletsInTime.txt",
    "amo": "AmoChainletsInTime.txt",
    "price": "pricedBitcoin2009-2018.csv",
}
GITHUB_RAW = "https://raw.githubusercontent.com/cakcora/BitcoinChainnet/master/data/"

# The prediction task, fixed once. December 2017 is the test period; September to November 2017 is the
# validation period used for every modelling choice.
HORIZON_DAYS = 1
TEST_START = pd.Timestamp("2017-12-01")
TEST_END = pd.Timestamp("2017-12-31")
VAL_START = pd.Timestamp("2017-09-01")
VAL_END = pd.Timestamp("2017-11-30")

# The chainlet matrix is 20 x 20; row and column 20 hold the capped cells.
N_CHAINLET = 20


def ensure_data_files(data_dir=DATA_DIR):
    """Return the paths of the three course files, downloading any that are missing."""
    os.makedirs(data_dir, exist_ok=True)
    paths = {}
    for key, name in FILES.items():
        path = os.path.join(data_dir, name)
        if not os.path.exists(path):
            # Download under a temporary name so an interrupted run never leaves a half-written file.
            partial = path + ".part"
            urllib.request.urlretrieve(GITHUB_RAW + name, partial)
            os.replace(partial, path)
        paths[key] = path
        print("%-5s %s  %.1f MB" % (key, path, os.path.getsize(path) / 1e6))
    return paths


PATHS = ensure_data_files()

occ   data/OccChainletsInTime.txt  3.0 MB
amo   data/AmoChainletsInTime.txt  8.0 MB
price data/pricedBitcoin2009-2018.csv  0.1 MB


## 1. Load data and parse dates

## 2. Merge occurrence, amount and price

## 3. Feature engineering

## 4. Split and scaling

## 5. Model selection

## 6. Final evaluation on December 2017

## 7. Explaining the model with SHAP

## 8. Predictions file

## 9. Compliance audit